# 教師なし学習

## アソシエーション分析（バスケット分析）

### データセットの読み込み
- 商品購買のデータを `basket_data.csv` から読み込む
  - `date`: 日付（分析には使用しない）
  - `customer_id`: 購入者
  - `item_name`: 購入商品

In [ ]:
# 商品購買のデータ (cf. https://www.kaggle.com/datasets/acostasg/random-shopping-cart)
import pandas as pd
df = pd.read_csv("basket_data.csv")
display(df)

### データ形式の変換
- 購入者 (customer_id) ごとに，購入商品 (item_name) をまとめる

In [ ]:
# customer_id 列が同じ値の行について，item_name 列の値をまとめて list にする
# (index が customer_id，値が item_name の値リストの index 付き Series になる)
dataset = df.groupby("customer_id")["item_name"].apply(list)
display(dataset)

- 購入者を行，購入商品を列とする表を作り，値の True, False で誰が何を購入したかを表現する

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()

# 商品の一覧を抽出し(.fit(dataset))，2次元配列(ndarray)に変換(.transform(dataset))
#  - 各行は購入者
#  - 各列は購入商品
#  - 表の値は，購入していれば `True`，購入していなければ `False`
te_ary = te.fit(dataset).transform(dataset)

# 更に，インデックス(customer_id)と列名の付いたデータフレーム形式に変換
df2 = pd.DataFrame(te_ary, columns=te.columns_, index=dataset.index)
display(df2)

### 支持度の計算
- 支持度(support)は，購入者の，顧客全体に対する比率を表す
- ここでは，購入商品の組み合わせごとに支持度(support)を計算して表にする
- すべての組み合わせは莫大な数になるので，支持度(support)が0.04(4%)以上のものに限定している

In [ ]:
from mlxtend.frequent_patterns import apriori
frequet_itemsets = apriori(df2, min_support=0.04, use_colnames=True)
display(frequet_itemsets)

### アソシエーション分析の実行
- 前セルの支持度(support)の表をもとに，アソシエーション分析を行う
- 各行(アソシエーション・ルール)は，前提(antecedents)の商品を買った（以下事象$A$）人が結果(consequents)の商品を買う（以下，事象$C$）かどうかの情報を表す．
  各列の値は以下の通り
  - `antecedent support`: 前提の支持度．前提の商品の購入者の，顧客全体に対する比率．$P(A)$．前提の商品がどのくらい売れているか
  - `consequent support`: 結果の支持度．結果の商品の購入者の，顧客全体に対する比率．$P(C)$．結果の商品がどのくらい売れているか
  - `support`: 支持度．前提と結果の両方の商品を同時に買った人の，顧客全体に対する比率．$P(A∩C)$．前提と結果の両方の商品を同時に買った顧客がどのくらいいるか
  - `confidence`: 信頼度．$\frac{P(A∩C)}{P(A)}=P(C|A)$，前提の商品の購入者のうち，結果の商品を買った顧客はどのくらいいるか
  - `lift`: リフト値．信頼度の結果の支持度に対する比率．$\frac{P(C|A)}{P(C)}$．前提の商品を買ったという条件が加わると結果の商品を買う確率が何倍になるかということ
  - `leverage`: 影響度．支持度から前提の支持度と結果の支持度の積を引いたもの．$P(A∩C)-P(A)P(C)$．前提事象$A$と結果事象$C$の両方が起こる確率について，実際の確率と $A,C$が独立であると仮定したときの確率との差
  - `conviction`: 確信度．$\frac{1-P(C)}{1-P(C|A)}=\frac{P(\overline C)}{P(\overline C|A)}$．前提の商品を買ったという条件が加わると結果の商品を買わないという確率が何分の1になるかということ
- 前提と結果は複数の商品の組み合わせもあり，組み合わせの数が膨大になるので，リフト値(lift)が1以上の組み合わせに限定している


In [ ]:
from mlxtend.frequent_patterns import association_rules

# アソシエーション分析の実行（アソシエーション・ルールの抽出）
rules = association_rules(frequet_itemsets, metric="lift", min_threshold=1)

# 支持度(support)の降順に並べ替える
rules = rules.sort_values("support", ascending=False)

# 先頭の20行のみ表示
display(rules.head(20))

## クラスタリング
- 多数の参加者に，自分にとって「かわいいもの」と，それらに 17個の形容語がどのくらい当てはまるかを -2～2 の5段階で評価してもらったデータを元に，「かわいいもの」を2つのクラスタに分けてみる (2012年に調査したデータを一部抜粋・修正)
- クラスタリングの方法は，ユークリッド距離を使った k-means 法を用いる．

### データの読み込み

In [ ]:
import pandas as pd

# kawaii.csv を読み込んで表示
kawaii = pd.read_csv("kawaii.csv", )
display(kawaii)

### クラスタリングの実行
- 形容語のデータを元に，2つのクラスタに分ける

In [ ]:
# KMeans の memory leak の回避
import os
os.environ["OMP_NUM_THREADS"] = "1"

# 形容語の評定値のデータを kawaii_feat に代入
kawaii_feat = kawaii.drop(["評定者", "対象"], axis=1)

# kawaii_feat を用いてクラスタリングを実行(クラスタ数2)
from sklearn.cluster import KMeans
kawaii_cluster = KMeans(2, random_state=0).fit(kawaii_feat)

print("完了")

### 各クラスタの重心の表示
- 各クラスタに属する対象の，形容語ごとの評定値の平均 (= 重心) を求める  
  → 各クラスタの特徴を表す (-2: 当てはまらない，0: どちらともいえない, 2: 当てはまる)

In [ ]:
# 各クラスタの重心 (評定値の平均) の表示
centroid = kawaii_cluster.cluster_centers_
df_centroid = pd.DataFrame(centroid.transpose(), index=kawaii_feat.columns)
display(df_centroid)

### データに列を追加する
- `cluster`: 各対象が属するクラスタ
- `dist0`: この対象 と クラスタ0の重心 とのユークリッド距離
- `dist1`: この対象 と クラスタ1の重心 とのユークリッド距離

In [ ]:
# 元のデータ(kawaii)に cluster, dist0, dist1 の列を追加
from math import sqrt
kawaii["cluster"] = kawaii_cluster.labels_
kawaii["dist0"] = ((kawaii_feat - centroid[0])**2).apply(sum, axis = 1).map(sqrt)
kawaii["dist1"] = ((kawaii_feat - centroid[1])**2).apply(sum, axis = 1).map(sqrt)

# 結果の表を表示
display(kawaii)

### 各クラスタに属する対象を表示
- 各クラスタごとに，重心に近い方から30個を表示

In [ ]:
# 各クラスターの対象の，重心に近い方から30個を表示
print("クラスター0: ", kawaii[kawaii["cluster"]==0].sort_values("dist0").head(30)["対象"].values)
print("クラスター1: ", kawaii[kawaii["cluster"]==1].sort_values("dist1").head(30)["対象"].values)